# Multi-Label CNN Classifier

**1D Convolutional backbone for MALDI-TOF spectra, with 10 multi-label output heads.**

CNN exploits local peak structure and translation invariance in mass spectra. Three Conv1D layers (64→128→256 channels) with global average pooling, followed by a small MLP head and 10 independent sigmoid output heads.

**Architecture:** `Conv1D(k=7,c=64) → MaxPool → Conv1D(k=7,c=128) → MaxPool → Conv1D(k=5,c=256) → GlobalAvgPool → Linear(256→128) → 10×Linear(128→1)`

**Compared against:** OneVsRest Logistic Regression baseline on the same aggregated 70/15/15 split.

**Loss:** Masked BCE (only labeled drugs contribute per sample).

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    DATA_ROOT = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed")

OUT_DIR = Path("./results_multilabel_cnn")
OUT_DIR.mkdir(exist_ok=True)

BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]
SITES = ["DRIAMS-A", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]

print(f"Drugs: {len(DRUGS_10)}  |  Sites: {SITES}")

In [ ]:
# =============================================================================
# 1. MULTI-LABEL DATASET (same as 04-MultiLabel)
# =============================================================================

class MultiLabelDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(np.nan_to_num(Y, nan=0.0), dtype=torch.float32)
        self.mask = torch.tensor(~np.isnan(Y), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.mask[idx]


def load_multilabel_split():
    """Build multi-label X/Y from Processed per-drug CSVs. A = train, B/C/D = test."""
    train_X, train_Y, train_sp = None, None, None
    test_sites = {}

    for site_name in SITES:
        proc_site = f"Proc_{site_name}"
        site_labels = {}; X_samples = {}; sp_samples = {}; site_drugs_ok = []

        for drug_name in DRUGS_10:
            p = DATA_ROOT / proc_site / drug_name / "data.csv"
            if not p.exists(): continue
            df = pd.read_csv(p)
            site_drugs_ok.append(drug_name)
            for _, row in df.iterrows():
                code = row["code"]
                if code not in X_samples:
                    X_samples[code] = row[BIN_COLS].values.astype("float32")
                    sp_samples[code] = row["species"]
                site_labels.setdefault(code, {})[drug_name] = row["label"]

        if not site_drugs_ok: continue
        codes = list(X_samples.keys())
        X = np.array([X_samples[c] for c in codes], dtype="float32")
        species = np.array([sp_samples[c] for c in codes])

        Y = np.full((len(codes), len(DRUGS_10)), np.nan, dtype=float)
        for i, drug in enumerate(DRUGS_10):
            for j, code in enumerate(codes):
                if drug in site_labels.get(code, {}):
                    Y[j, i] = site_labels[code][drug]

        n_lab = (~np.isnan(Y)).sum(axis=1).mean()
        print(f"  {site_name}: {len(codes)} samples, avg {n_lab:.1f} labels/sample")

        if site_name == "DRIAMS-A":
            train_X, train_Y, train_sp = X, Y, species
        else:
            test_sites[site_name] = (X, Y, species)

    print(f"\nTrain (A): {train_X.shape[0]} samples, {len(np.unique(train_sp))} species")
    for s, (Xs, Ys, sps) in test_sites.items():
        print(f"  Test {s}: {Xs.shape[0]} samples, {len(np.unique(sps))} species")
    return train_X, train_Y, train_sp, test_sites

---
## CNN Architecture + MultiLabelCNN Class

In [ ]:
# =============================================================================
# 2. CNN BACKBONE ARCHITECTURE
# =============================================================================

class SpectralCNN(nn.Module):
    """1D CNN backbone for MALDI-TOF spectra.

    Architecture:
        Input (6000,) -> Conv1D(k=7,c=64) -> MaxPool(2)  -- 3000
                      -> Conv1D(k=7,c=128) -> MaxPool(2) -- 1500
                      -> Conv1D(k=5,c=256) -> GlobalAvgPool -- 256
                      -> Linear(256->128) -> 10 x Linear(128->1) + Sigmoid
    """
    def __init__(self, input_dim=6000, n_labels=10, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2))
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256), nn.ReLU())

        # MLP head after global pooling
        self.mlp = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout))

        # 10 independent output heads
        self.heads = nn.ModuleList([nn.Linear(128, 1) for _ in range(n_labels)])

    def forward(self, x):
        # x: (B, 6000) -> (B, 1, 6000)
        x = x.unsqueeze(1)
        x = self.conv1(x)      # (B, 64, 3000)
        x = self.conv2(x)      # (B, 128, 1500)
        x = self.conv3(x)      # (B, 256, 1500)
        x = x.mean(dim=2)      # Global Avg Pool -> (B, 256)
        x = self.mlp(x)        # (B, 128)
        return torch.cat([h(x) for h in self.heads], dim=1)  # (B, 10)


class MultiLabelCNN(BaseEstimator, ClassifierMixin):
    """sklearn-compatible multi-label CNN classifier."""
    def __init__(self, dropout=0.3, learning_rate=1e-3, weight_decay=0.0,
                 batch_size=64, epochs=50, early_stopping_patience=10,
                 warmup_epochs=0, val_fraction=0.1,
                 random_state=42, verbose=False, device="cpu"):
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.early_stopping_patience = early_stopping_patience
        self.warmup_epochs = warmup_epochs
        self.val_fraction = val_fraction
        self.random_state = random_state
        self.verbose = verbose
        self.device = device

    def fit(self, X, Y):
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)
        self.n_labels_ = Y.shape[1]
        self.model_ = SpectralCNN(n_labels=self.n_labels_, dropout=self.dropout).to(self.device)

        ds = MultiLabelDataset(X, Y)
        n_val = max(1, int(len(ds) * self.val_fraction))
        n_tr = len(ds) - n_val
        train_ds, val_ds = random_split(ds, [n_tr, n_val],
            generator=torch.Generator().manual_seed(self.random_state))
        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=self.batch_size * 2, shuffle=False)

        opt_cls = torch.optim.AdamW if self.weight_decay > 0 else torch.optim.Adam
        optimizer = opt_cls(self.model_.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        warmup = max(0, self.warmup_epochs)
        t_max = max(1, self.epochs - warmup)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=1e-6)
        criterion = nn.BCEWithLogitsLoss(reduction="none")

        best_val_loss = float("inf"); best_state = None; patience_counter = 0

        for epoch in range(self.epochs):
            self.model_.train()
            train_loss = 0.0
            for xb, yb, mb in train_loader:
                xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                if epoch < warmup:
                    for pg in optimizer.param_groups:
                        pg["lr"] = self.learning_rate * (epoch + 1) / warmup
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss = criterion(logits, yb)
                loss = (loss * mb).sum() / mb.sum()
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            if epoch >= warmup: scheduler.step()

            self.model_.eval(); val_loss = 0.0
            with torch.no_grad():
                for xb, yb, mb in val_loader:
                    xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    val_loss += loss.item()
            val_loss /= len(val_loader); train_loss /= len(train_loader)

            if self.verbose:
                print(f"  Epoch {epoch+1:3d}/{self.epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    if self.verbose: print(f"  Early stopping at epoch {epoch+1}")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.model_.eval()
        self.is_fitted_ = True
        return self

    def predict_proba(self, X):
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            return torch.sigmoid(self.model_(X_t)).cpu().numpy()

    def predict(self, X, thresholds=None):
        proba = self.predict_proba(X)
        if thresholds is None: thresholds = [0.5] * self.n_labels_
        return (proba >= np.array(thresholds)).astype(int)

---
## Load Aggregated Data + Species-Stratified Split

In [ ]:
# =============================================================================
# 3. LOAD AGGREGATED DATA + SPLIT (70/15/15)
# =============================================================================

print("Loading aggregated data from all 4 sites...")
X_train_raw, Y_train_raw, sp_train_raw, test_data_raw = load_multilabel_split()

# Concatenate all sites
X_all = np.concatenate([X_train_raw] + [v[0] for v in test_data_raw.values()])
Y_all = np.concatenate([Y_train_raw] + [v[1] for v in test_data_raw.values()])
sp_all = np.concatenate([sp_train_raw] + [v[2] for v in test_data_raw.values()])
print(f"Total aggregated: {X_all.shape[0]} samples, {len(np.unique(sp_all))} species")

# Species-stratified split: train 70% / val 15% / test 15%
idx_trval, idx_test, _, _ = stratified_species_drug_split(
    np.arange(len(Y_all)).reshape(-1, 1),
    np.zeros(len(Y_all)),
    species=sp_all, test_size=0.15, random_state=SEED)
idx_trval = idx_trval.flatten().astype(int); idx_test = idx_test.flatten().astype(int)

X_trval, Y_trval, sp_trval = X_all[idx_trval], Y_all[idx_trval], sp_all[idx_trval]
X_test, Y_test = X_all[idx_test], Y_all[idx_test]

val_frac = 0.15 / 0.85
idx_train, idx_val, _, _ = stratified_species_drug_split(
    np.arange(len(Y_trval)).reshape(-1, 1),
    np.zeros(len(Y_trval)),
    species=sp_trval, test_size=val_frac, random_state=SEED)
idx_train = idx_train.flatten().astype(int); idx_val = idx_val.flatten().astype(int)
X_train, Y_train = X_trval[idx_train], Y_trval[idx_train]
X_val, Y_val = X_trval[idx_val], Y_trval[idx_val]

print(f"Split: train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

# Preprocess
state = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state); X_val_pp = apply_input_transform(X_val, state)
X_test_pp = apply_input_transform(X_test, state)
print("Preprocessing done.")

---
## Baseline: OneVsRest LR

In [ ]:
# =============================================================================
# 4. BASELINE: OneVsRest LR (aggregated, 70/15/15)
# =============================================================================

C_grid = np.linspace(5e-5, 1e-3, 15)
thresholds = np.linspace(0.05, 0.95, 91)

lr_baseline = {}
for di, drug in enumerate(tqdm(DRUGS_10, desc="LR per drug")):
    tr_mask = ~np.isnan(Y_train[:, di])
    X_tr, y_tr = X_train_pp[tr_mask], Y_train[tr_mask, di].astype(int)
    val_mask = ~np.isnan(Y_val[:, di])
    X_v, y_v = X_val_pp[val_mask], Y_val[val_mask, di].astype(int)
    if len(np.unique(y_tr)) < 2: continue

    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
        param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
    grid.fit(X_tr, y_tr)
    lr = grid.best_estimator_
    proba_v = lr.predict_proba(X_v)[:, 1]
    best_t = thresholds[np.argmax([balanced_accuracy_score(y_v, proba_v >= t) for t in thresholds])]

    # Test
    ts_mask = ~np.isnan(Y_test[:, di])
    if ts_mask.sum() >= 2:
        y_ts = Y_test[ts_mask, di].astype(int)
        preds = lr.predict_proba(X_test_pp[ts_mask])[:, 1] >= best_t
        lr_baseline[drug] = balanced_accuracy_score(y_ts, preds)
    else:
        lr_baseline[drug] = np.nan

print("\nLR Baseline (test set):")
for d, ba in lr_baseline.items():
    print(f"  {d:35s}  {ba:.4f}")

---
## CNN Grid Search (dropout × lr)

In [ ]:
# =============================================================================
# 5. CNN GRID SEARCH (dropout x lr)
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

LR_GRID = np.linspace(1e-4, 5e-4, 6)
DROP_GRID = np.linspace(0.1, 0.5, 6)
print(f"Grid: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")

best_balacc = -1; best_lr = None; best_drop = None

for lr in LR_GRID:
    for d in DROP_GRID:
        cnn = MultiLabelCNN(
            dropout=d, learning_rate=lr, weight_decay=1e-3,
            batch_size=64, epochs=50, early_stopping_patience=10, warmup_epochs=10,
            val_fraction=0.1, random_state=SEED, verbose=False, device=device)
        cnn.fit(X_train_pp, Y_train)

        proba_v = cnn.predict_proba(X_val_pp)
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))
        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        mark = " *" if macro_ba > best_balacc else ""
        print(f"  lr={lr:.1e}  drop={d:.2f}  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > best_balacc:
            best_balacc = macro_ba; best_lr = lr; best_drop = d

print(f"\nBest: lr={best_lr:.1e} dropout={best_drop:.2f}  macro_BalAcc={best_balacc:.4f}")

---
## Best CNN + Threshold Tuning

In [ ]:
# =============================================================================
# 6. RETRAIN BEST CNN + THRESHOLD TUNING
# =============================================================================

cnn_best = MultiLabelCNN(
    dropout=best_drop, learning_rate=best_lr, weight_decay=1e-4,
    batch_size=64, epochs=100, early_stopping_patience=15, warmup_epochs=10,
    val_fraction=0.1, random_state=SEED, verbose=True, device=device)
cnn_best.fit(X_train_pp, Y_train)

# Per-drug thresholds on validation
proba_val = cnn_best.predict_proba(X_val_pp)
cnn_thresholds = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2: cnn_thresholds.append(0.5); continue
    pv = proba_val[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    cnn_thresholds.append(bt)
cnn_thresholds = np.array(cnn_thresholds)
print(f"\nPer-drug thresholds: {[f'{t:.2f}' for t in cnn_thresholds]}")

# Evaluate on test set
proba_test = cnn_best.predict_proba(X_test_pp)
cnn_results = {}
for di, drug in enumerate(DRUGS_10):
    tm = ~np.isnan(Y_test[:, di])
    if tm.sum() < 2: cnn_results[drug] = np.nan; continue
    preds = (proba_test[tm, di] >= cnn_thresholds[di])
    yt = Y_test[tm, di].astype(int)
    cnn_results[drug] = balanced_accuracy_score(yt, preds)

print("\nCNN Test BalAcc:")
for drug, ba in cnn_results.items():
    print(f"  {drug:35s}  {ba:.4f}")

---
## LR vs CNN Comparison

In [ ]:
# =============================================================================
# 7. LR vs CNN vs MLP COMPARISON
# =============================================================================

# Note: MLP results loaded from 04-MultiLabel aggregated run (mlp_agg_results)
# If running standalone, uncomment the MLP training block below or paste results

# Example MLP results from 04 run -- replace with actual values
_mlp_results = {}  # Populate with mlp_agg_results from 04 notebook

rows = []
for drug in DRUGS_10:
    rows.append({
        "Drug": drug[:15],
        "LR_OneVsRest": lr_baseline.get(drug, np.nan),
        "CNN_Shared": cnn_results.get(drug, np.nan),
    })
df_comp = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_comp)); w = 0.35
ax.bar(x - w/2, df_comp["LR_OneVsRest"], w, label="OneVsRest LR", color="#aec7e8")
ax.bar(x + w/2, df_comp["CNN_Shared"], w, label="Shared CNN", color="#1f77b4")
ax.set_xticks(x); ax.set_xticklabels(df_comp["Drug"], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Test BalAcc"); ax.set_title("LR vs CNN — Aggregated 70/15/15")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "cnn_vs_lr.pdf")
plt.show()

print("\nLR vs CNN comparison:")
print(df_comp.to_string(index=False))

# CNN heatmap
cnn_heatmap = pd.DataFrame([cnn_results], index=["CNN"]).T
cnn_heatmap.columns = ["Test_BalAcc"]
print("\nCNN Per-Drug Test BalAcc:")
print(cnn_heatmap.to_string())

In [ ]:
print("\nDone. CNN results saved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.pdf")):
    print(f"  {f.name}")